In [1]:
# =============================================================================
# Projeto: Análise de Recomendações Pedagógicas para o Uso de Inteligência
#          Artificial Generativa na Educação a Distância por meio de PLN e LLMs
# Disciplina: Inteligência Artificial - 7º N - Ciência da Computação
# Universidade Presbiteriana Mackenzie - Prof. Dr. Ivan Carlos Alcântara de Oliveira
# Integrante: Daniel Ribeiro Pauleti - RA 10723162 - 10723162@mackenzista.com.br
#
# Arquivo: notebooks/01_coleta_openalex.ipynb
# Síntese: coleta dos metadados de publicações (2023-2026) sobre IA generativa na
#          educação a distância por meio da API pública do OpenAlex.
#
# Histórico de alterações:
#   2026-09-25 - Daniel Ribeiro Pauleti - Versão inicial.
# =============================================================================

# 01 — Coleta de dados no OpenAlex

O OpenAlex é um catálogo aberto (licença CC0) de publicações científicas. As consultas combinam termos de **IA generativa** com termos de **educação a distância/online** e são aplicadas ao título e ao resumo (`title_and_abstract.search`), restritas ao período **2023–2026**. Para preservar a reprodutibilidade, a coleta só é refeita se `COLETAR = True`; caso contrário, usa-se o arquivo bruto versionado em `data/raw/`.

In [2]:
import sys, warnings
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))
warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.max_colwidth", 90)
import json
from coleta_openalex import CONSULTAS, main as coletar_tudo
COLETAR = False   # True para refazer a coleta (os resultados podem mudar com o tempo)
if COLETAR:
    coletar_tudo()
print(json.dumps(json.loads((RAIZ / "data/raw/consultas.json").read_text(encoding="utf-8")), ensure_ascii=False, indent=2))

{
  "anos": "2023-2026",
  "consultas": {
    "Q1": "(\"generative artificial intelligence\" OR \"generative AI\" OR ChatGPT OR \"large language model\" OR \"large language models\" OR GPT-4) AND (\"distance education\" OR \"distance learning\" OR \"open and distance learning\")",
    "Q2": "(\"generative artificial intelligence\" OR \"generative AI\" OR ChatGPT OR \"large language model\" OR \"large language models\" OR GPT-4) AND (\"online learning\" OR \"online education\") AND \"higher education\"",
    "Q3": "(\"inteligência artificial generativa\" OR \"IA generativa\" OR ChatGPT OR \"inteligencia artificial generativa\") AND (\"educação a distância\" OR \"ensino a distância\" OR \"educación a distancia\" OR EaD)"
  },
  "data_coleta": "2026-09-25"
}


In [3]:
bruto = pd.read_json(RAIZ / "data/raw/openalex_bruto.jsonl", lines=True)
print("Registros brutos (com repetição entre consultas):", len(bruto))
print("Registros únicos por ID OpenAlex:", bruto["id"].nunique())
bruto.groupby("_consulta")["id"].count().rename("registros").to_frame()

Registros brutos (com repetição entre consultas): 480
Registros únicos por ID OpenAlex: 471


,registros
_consulta,
Q1,256
Q2,167
Q3,57


In [4]:
# Sobreposição entre consultas
por_id = bruto.groupby("id")["_consulta"].apply(lambda s: "+".join(sorted(set(s))))
por_id.value_counts().rename("registros").to_frame()

,registros
_consulta,
Q1,247
Q2,158
Q3,57
Q1+Q2,9


In [5]:
# Exemplo de registro bruto: o resumo vem como índice invertido {palavra: [posições]}
ex = bruto.iloc[0]
print(ex["title"])
print(list(ex["abstract_inverted_index"].items())[:8])

Generative Artificial Intelligence in distance education: Transformations, challenges, and impact on academic integrity and student voice
[('Generative', [0]), ('artificial', [1]), ('intelligence', [2]), ('(GenAI)', [3]), ('has', [4]), ('reshaped', [5]), ('distance', [6, 29, 51, 66, 89, 157]), ('education', [7, 158])]
